In [2]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# =========================
# 1. Load data
# =========================
df = pd.read_csv("hasil_labeling_ev - NEGATIF 800-2.csv")
df["clean_textdisplay"] = df["clean_textdisplay"].astype(str).apply(clean_text)

docs = df["clean_textdisplay"].fillna("").astype(str).tolist()

/Users/mthariqsultand/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
print(df.columns)

Index(['date', 'authordisplayname', 'textdisplay', 'likecount',
       'clean_textdisplay', 'label', 'confidence', 'status', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19'],
      dtype='object')


In [4]:
import re

# =========================
# 1. CUSTOM STOPWORDS (ISI SENDIRI)
# =========================
custom_stopwords = set([
    # isi sendiri nanti
    "yang",
    "dan",
    "di",
    "ke",
    "dari",
    "ini",
    "itu",
    "gak",
    "ga",
    "saya",
    "kamu",
    "dia",
    "kami",
    "mereka",
    "ya",
    "ad",
    "jual",
    "dan",
    "dari",
    "yg",
    "byk",
    "query",
    "nya",
])

# =========================
# 2. CLEAN FUNCTION
# =========================
def clean_text(text):
    text = str(text).lower()

    # hapus url
    text = re.sub(r"http\S+|www\S+", "", text)

    # hapus mention & hashtag
    text = re.sub(r"@\w+|#\w+", "", text)

    # hapus angka & simbol
    text = re.sub(r"[^a-z\s]", " ", text)

    # rapihin spasi
    text = re.sub(r"\s+", " ", text).strip()

    # hapus stopwords custom
    words = text.split()
    words = [w for w in words if w not in custom_stopwords]

    return " ".join(words)

In [5]:
# =========================
# 2. SOTA EMBEDDING MODEL
# =========================
model = SentenceTransformer("intfloat/multilingual-e5-large")

# E5 BUTUH PREFIX (IMPORTANT!)
embeddings = model.encode(
    ["query: " + d for d in docs],   # hanya untuk embedding
    show_progress_bar=True
)


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

In [6]:
# =========================
# 3. REDUCE DIMENSION (UMAP)
# =========================
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [7]:
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [8]:
# =========================
# 5. BERTopic MODEL
# =========================
topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-06-10 18:52:43,424 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-10 18:52:46,727 - BERTopic - Dimensionality - Completed ✓
2026-06-10 18:52:46,727 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-10 18:52:46,737 - BERTopic - Cluster - Completed ✓
2026-06-10 18:52:46,738 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-10 18:52:46,752 - BERTopic - Representation - Completed ✓


In [9]:
# =========================
# 6. TRAIN MODEL
# =========================
topics, probs = topic_model.fit_transform(docs)

2026-06-10 18:52:46,768 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

2026-06-10 18:52:52,596 - BERTopic - Embedding - Completed ✓
2026-06-10 18:52:52,597 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-10 18:52:53,413 - BERTopic - Dimensionality - Completed ✓
2026-06-10 18:52:53,413 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-10 18:52:53,421 - BERTopic - Cluster - Completed ✓
2026-06-10 18:52:53,422 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-10 18:52:53,437 - BERTopic - Representation - Completed ✓


In [10]:
topic_model.visualize_topics()

In [11]:
topic_model.visualize_barchart(top_n_topics=10)

In [12]:
# =========================
# 8. TOPIC SUMMARY
# =========================
print(topic_model.get_topic_info())


    Topic  Count                                  Name  \
0      -1    208     -1_mobil_vehicle_electric_listrik   
1       0    105                0_mobil_beli_harga_ada   
2       1     96           1_baterai_harga_mobil_tahun   
3       2     68                2_brio_kuda_suzuki_aja   
4       3     56        3_electric_vehicle_air_listrik   
5       4     47           4_bensin_bakar_minyak_bahan   
6       5     47               5_harga_juta_brio_kalau   
7       6     31  6_listrik_merusak_lingkungan_tambang   
8       7     29             7_jepang_china_cina_mobil   
9       8     27           8_indonesia_rata_bumn_motor   
10      9     23       9_pemerintah_bantuan_pajak_ubah   
11     10     22       10_hidrogen_hydrogen_energi_air   
12     11     14         11_ice_electric_vehicle_mobil   
13     12     14     12_charger_portable_wall_charging   
14     13     13           13_listrik_mobil_dokter_doi   

                                       Representation  \
0   [mobil, ve

In [13]:
df_pred = pd.read_csv("hasil_sentimen.predictions - negative.csv")

new_docs = df_pred["clean_textdisplay"].astype(str).tolist()

In [14]:
pred_topics, pred_probs = topic_model.transform(new_docs)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

2026-06-10 18:52:54,508 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-10 18:52:55,914 - BERTopic - Dimensionality - Completed ✓
2026-06-10 18:52:55,915 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-10 18:52:55,922 - BERTopic - Cluster - Completed ✓


In [15]:
df_pred["pred_topic"] = pred_topics

In [17]:
topic_info = topic_model.get_topic_info()
topic_names = dict(zip(topic_info["Topic"], topic_info["Name"]))
df_pred["topic_name"] = df_pred["pred_topic"].map(topic_names)


In [18]:
print(df_pred.head())

                        date     authordisplayname  \
0  2024-10-26 01:55:33+00:00               @kedepp   
1  2024-10-26 01:02:47+00:00  @adolfbabehdolof8546   
2  2024-10-25 23:48:20+00:00              @guz3108   
3  2024-10-26 01:27:38+00:00      @aanganggara1699   
4  2024-10-25 23:12:27+00:00    @wawansetiawan3474   

                                         textdisplay  likecount  \
0  Jikanper hari isi bensin 100ribu, setahun 360 ...        0.0   
1  Konten negatif, selalu menjelekkan EV, karena ...        6.0   
2  Harga jual itu soal supply demand. Sekarang ke...        1.0   
3  Anjiir murah amat klo 500rb sebulan, bensin mo...        0.0   
4  Gak fair sih klo penilaian hanya di harga jual...        2.0   

                                   clean_textdisplay    label  confidence  \
0  jikanper hari isi bensin ribu setahun hari ber...  Positif        0.86   
1  konten negatif selalu menjelekkan electric veh...  Positif        0.91   
2  harga jual itu soal supply demand seka

In [19]:
df_pred.to_csv("hasil_prediksi_topik.csv", index=False)